# Unit 14 — Two Pointers & Sliding Window

A school store has a SORTED list of prices and a fixed budget: can two prices spend it exactly without testing every pair? And what is the longest run of days whose total training stays under a limit? Both become LINEAR when two indices move forward with a clear rule. We build it as a short ladder of executable demos with a **Notice**, then a full stdin solver.

## Lesson 1 — Converging Two Pointers

On a SORTED list, start `lo` at the front and `hi` at the back. If the pair sum is too small, move `lo` up; too big, move `hi` down; equal, done.

In [ ]:
values = [1, 3, 5, 8, 12]
target = 13
lo = 0
hi = 4
found = False
while lo < hi and found == False:
    total = values[lo] + values[hi]
    print("check", values[lo], "+", values[hi], "=", total)
    if total == target:
        found = True
    elif total < target:
        lo = lo + 1
    else:
        hi = hi - 1
print(found)

**Notice:** each step rules out one value from an END, so the pair `1 + 12 = 13` is found in a few checks — never all pairs.

The same sweep can track the CLOSEST sum instead of an exact match.

In [ ]:
values = [1, 4, 6, 9]
target = 12
lo = 0
hi = 3
best = 999
while lo < hi:
    total = values[lo] + values[hi]
    gap = abs(total - target)
    if gap < best:
        best = gap
    if total < target:
        lo = lo + 1
    else:
        hi = hi - 1
print("closest gap", best)

**Notice:** keep the smallest gap seen while converging; here the closest sum to 12 is off by 1.

Why is it `O(n)`? Each pointer only ever moves ONE way, so together they take at most `n` steps — count them.

In [ ]:
values = [2, 4, 6, 8, 10, 12]
target = 100
lo = 0
hi = 5
moves = 0
while lo < hi:
    total = values[lo] + values[hi]
    if total < target:
        lo = lo + 1
    else:
        hi = hi - 1
    moves = moves + 1
print("pointer moves:", moves, "for n =", len(values))

**Notice:** the two pointers make `n - 1 = 5` moves total before meeting — linear, not quadratic.

**Put it together:** the program reads `N`, a `target`, then the `N` ALREADY-SORTED values, and prints `YES` if two of them sum to the target, else `NO`.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
target = int(tokens[1])
values = []
i = 0
while i < n:
    values.append(int(tokens[i + 2]))
    i = i + 1

answer = "NO"
lo = 0
hi = n - 1
while lo < hi and answer == "NO":
    pair_sum = values[lo] + values[hi]
    if pair_sum == target:
        answer = "YES"
    elif pair_sum < target:
        lo = lo + 1
    else:
        hi = hi - 1
print(answer)


Run the full solver from this unit folder:

```text
python assets/l1.py < assets/l1/1.in
```

**Notice:** converge `lo`/`hi` on the sorted list; a `found`-style flag stops the loop when the pair sum equals the target.

**Complexity:** `O(n)` — the input is already sorted, so no sort is needed.

## Lesson 2 — Sliding Windows

A window `[left, right]` grows by adding `values[right]`; keep a RUNNING SUM instead of re-adding. This needs NON-NEGATIVE values so shrinking is safe.

In [ ]:
values = [4, 3, 1, 1, 1]
window_sum = 0
right = 0
while right < len(values):
    window_sum = window_sum + values[right]
    print("added", values[right], "-> window sum", window_sum)
    right = right + 1

**Notice:** `window_sum` accumulates as `right` advances — one add per step, no re-summing.

Shrink from the LEFT while the window is over budget; the widest window seen is the longest affordable stretch.

In [ ]:
values = [4, 3, 1, 1, 1]
limit = 6
left = 0
window_sum = 0
best = 0
right = 0
while right < len(values):
    window_sum = window_sum + values[right]
    while window_sum > limit and left <= right:
        window_sum = window_sum - values[left]
        left = left + 1
    length = right - left + 1
    if length > best:
        best = length
    right = right + 1
print("longest affordable window:", best)

**Notice:** when the sum exceeds 6, drop `values[left]` and advance `left`; the best width here is 3.

Reverse the goal: shrink while the window MEETS a target to find the SHORTEST qualifying window.

In [ ]:
values = [5, 1, 1, 1, 7]
target = 9
left = 0
window_sum = 0
shortest = len(values) + 1
right = 0
while right < len(values):
    window_sum = window_sum + values[right]
    while window_sum >= target and left <= right:
        length = right - left + 1
        if length < shortest:
            shortest = length
        window_sum = window_sum - values[left]
        left = left + 1
    right = right + 1
print("shortest reaching target:", shortest)

**Notice:** once the sum reaches 9, shrink to record the shortest length; here it is 3 (`1 1 7`).

**Put it together:** the program reads `N`, a `limit`, then the `N` non-negative values, and prints the LONGEST window whose sum stays `<= limit`.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
limit = int(tokens[1])
values = []
i = 0
while i < n:
    values.append(int(tokens[i + 2]))
    i = i + 1

left = 0
right = 0
window_sum = 0
best_length = 0
while right < n:
    window_sum = window_sum + values[right]
    while window_sum > limit and left <= right:
        window_sum = window_sum - values[left]
        left = left + 1
    current_length = right - left + 1
    if current_length > best_length:
        best_length = current_length
    right = right + 1
print(str(best_length))


Run the full solver from this unit folder:

```text
python assets/l2.py < assets/l2/1.in
```

**Notice:** grow the window with a running sum; while it exceeds `limit`, shrink from the left; track the widest width.

**Complexity:** `O(n)` — `left` and `right` each advance at most `n` times.

## Two-Pointer Checklist

(1) Converging pointers need a SORTED list; move the end that improves the sum; (2) a sliding window keeps a running sum and needs NON-NEGATIVE values to shrink safely; (3) longest vs shortest just flips the shrink condition; (4) each pointer moves one way → `O(n)`.